# Objetivo: que toda la difusion vaya junta, y que cambiando las clases cambiemos soii es mnist o de audios o stable

In [ ]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import time

import torchaudio.transforms as T
import math   
from src.dataset import NSynth
import torch
from src.diffusion import *
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
root = r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\diffusion"
paths = {
    'minst' : {
        'model': root + r"\minst\model.pth",
        'scheduler': root + r"\minst\scheduler.pth",
        'checkpoint': root + r"\minst\checkpoint.pth"
    },
    'audio' : {
        'model': root + r"\audio\model.pth",
        'scheduler': root + r"\audio\scheduler.pth",
        'checkpoint': root + r"\audio\checkpoint.pth"
    },
    'stable' : {
        'model': root + r"\stable\model.pth",
        'scheduler': root + r"\stable\scheduler.pth",
        'checkpoint': root + r"\stable\checkpoint.pth",
    }
}

In [ ]:
def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)

def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch'], checkpoint['loss']

In [ ]:
def pipeline_audio(x):
    wave, _, _, _ = x # solo nos quedamos con la onda, el resto no nos interesa
    sttft_transform = T.Spectrogram(n_fft=1024, hop_length=512)
    stft_spec = stft_transform(wave)
    log_mag, sin, cos = compute_magnitude_and_phase_sin_cos(stft_spec) ## SINCOS
    x = torch.cat([log_mag, sin, cos], dim=1).to(device)  ## SINCOS

In [ ]:
def train(
    model, 
    transform_data=None,
    epochs=100, 
    batch_size=16, 
    lr=1e-3, 
    model_path=None,
    sch_path=None, 
    dataset=NSynth('training'), 
    verb=True,
    verb_batch=False,
    type='audio',
):
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    scheduler = Scheduler(num_epochs=epochs).to(device)
    diffuser = Diffuser(model, scheduler).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)    
    scaler = torch.cuda.amp.GradScaler()
    
    mse_loss = nn.MSELoss()
    best_loss = 1
    losses = []
        
    for epoch in range(epochs):
        start_time = time.time() 
        _d_batch_index = 0
        epoch_loss = 0
        for x in train_loader:
            _d_batch_index += 1
            # wave = wave.to(device)
            # x = stft_transform(wave)
            x = x.to(device)
            if transform_data is not None:
                x = transform_data(x)
                
            batch_size = x.size(0)
            t = torch.randint(0, epochs, (batch_size,), device=device, dtype=torch.long)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(dtype=torch.bfloat16): # OPTIMIZACION FLOAT 32
                # print(f"mag: {mag.shape}, sin: {sin.shape}, cos: {cos.shape}, x: {x.shape}")
                z, e = diffuser(x, t)
                e_pred = model(z, t)
                if e_pred.shape != e.shape: # TODO borrar esto es debug
                    print('error en los tamaños')

                loss = mse_loss(e_pred, e)
                epoch_loss += loss.item()

            scaler.scale(loss).backward() # TODO no se si la funcion de loss es la mas optima para el caso
            scaler.step(optimizer)
            scaler.update()
            
            if verb_batch:
                if _d_batch_index %10 == 0:
                    _t = time.time() - start_time
                    print(f'tiempo en {_d_batch_index} batchs: {_t:.5f} \t| media de tiempo por batch: {_t/_d_batch_index:.5f} \t| media de loss: {epoch_loss/_d_batch_index:.5f}')

        losses.append(epoch_loss/_d_batch_index)
            
        if verb:
            _t = time.time() - start_time
            print(f"Epoch {epoch}, Loss: {loss.item()}, time: {time.time() - start_time}, avg wave time: {_t/(batch_size*_d_batch_index)}")
            
        if loss.item() < best_loss:
            best_loss = loss.item()
            
            if type:
                save_checkpoint(model, optimizer, epoch, loss.item(), paths[type]['checkpoint'])
            
    
    # AL FINALIZAR EL ENTRENAMIENTO, GUARDAMOS EL MODELO Y EL SCHEDULER
    if type:
        torch.save(model.state_dict(), paths[type]['model'])
        torch.save(scheduler.state_dict(), paths[type]['scheduler'])
        
    print("Training completed., best loss:", best_loss)
    
    return model, scheduler, losses
    